In [0]:
from pyspark.sql import SparkSession, DataFrame, Window
from datetime import datetime
from pyspark.sql.functions import to_timestamp, col, date_trunc, current_timestamp, to_date, concat_ws, sha2, row_number, trim, upper, when
from delta.tables import DeltaTable
import logging

In [0]:
def date_now() -> datetime:
    return datetime.strptime(datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "%Y-%m-%d %H:%M:%S")

In [0]:
dbutils.widgets.text("target_table_name", "", "Gold name")
dbutils.widgets.text("source_table_name", "", "Silver name")
dbutils.widgets.dropdown("environment", "dev", ["dev", "prod"])

gold_name = dbutils.widgets.get("target_table_name")
silver_name = dbutils.widgets.get("source_table_name")
environment = dbutils.widgets.get("environment")

In [0]:
table_comment = 'The table contains sales transaction data, including details about the products sold, customers, and the sales channel. It can be used for analyzing sales performance, tracking customer purchasing behavior, and understanding trends over time. Key fields include the sale date, total value of the sale, and identifiers for products and customers.'

columns_comments = {
        "cod_venda": "unique sales code",
        "cod_produto": "product code",
        "cod_cliente": "client code",
        "data_venda": "date of sale",
        "desc_canal": "desc of channel sale",
        "valor_venda_total": "value total sale",
        "chave": "sale hash primary key",
        "data_atualizacao": "sale update date"
    }

columns_silver_gold = {
        "id_venda": "cod_venda",
        "id_produto": "cod_produto",
        "id_cliente": "cod_cliente",
        "data_venda": "data_venda",
        "canal": "desc_canal",
        "valor_total": "valor_venda_total",
        "hash_key": "chave",
        "updated_at": "data_atualizacao"
    }

tags = {
        'responsible': 'commerce',
        'refresh_freq': 'D-1',
        'table_type': 'fato',
        'data_owner': 'Leonardo Nunes',
        'table': 'vendas'
    }

In [0]:
source_path = f'`comercial-{environment}`.`silver`.`{silver_name}`'
target_path = f'`comercial-{environment}`.`gold`.`{gold_name}`'
logs_path = f'`comercial-{environment}`.`corporate`.`logs`'

In [0]:
spark = SparkSession.builder \
    .appName("Ingestion from csv to delta") \
    .getOrCreate()

logging.basicConfig(
    level=logging.INFO
    , format="%(asctime)s | %(levelname)s | %(message)s"
    , datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

In [0]:
def read_silver() -> DataFrame: 
    df_bronze = spark.table(source_path)
    return df_bronze

In [0]:
def table_exists_in_catalog(table_name: str) -> bool:
    return spark.catalog.tableExists(table_name)

def get_delta_table(df_table_name: str) -> DeltaTable:
    return DeltaTable.forName(spark, df_table_name)

In [0]:
def apply_refactory_name_column(df_silver: DataFrame) -> DataFrame:
    df_fato_vendas = df_silver.select(
        *[
            col(column_silver).alias(column_gold) for column_silver, column_gold in columns_silver_gold.items()
        ]
    )
    return df_fato_vendas

def add_description_table():
    spark \
        .sql(f"""
            ALTER TABLE {target_path}
            SET TBLPROPERTIES (
                'comment' = '{table_comment}'
            )
              """)

def add_tag_table():
    for tag_name, tag_value in tags.items():
        spark \
            .sql(f"""
                ALTER TABLE {target_path}
                SET TBLPROPERTIES (
                    '{tag_name}' = '{tag_value}'
                )
                """)

def add_comment_column(columns_comments: dict):
    for column_name, column_comment in columns_comments.items():
        spark.sql(f"""
            ALTER TABLE {target_path}
            ALTER COLUMN {column_name} COMMENT '{column_comment}'
        """)

In [0]:
logs = {
    "dat_Hr_Process_Start": date_now(),
    "dat_Hr_Process_End":  '1900-01-01 00:00:00',
    "layer": "GOLD",
    "catalog": f'comercial-{environment}'.upper(),
    "source_Name": silver_name.upper(),
    "target_Name": gold_name.upper(),
    "status": None,
    "count_Rows": None,
}

In [0]:
def merge_fato_vendas(silver: DataFrame, columns_comments: dict):
    if table_exists_in_catalog(target_path):
        get_delta_table(target_path) \
            .alias("target") \
            .merge(
                silver.alias("source"),
                "source.chave = target.chave"
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
    else: 
        silver.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_path)

        logger.info("Start add documentation table")
        add_description_table()
        add_tag_table()
        add_comment_column(columns_comments)
        logger.info("End add documentation table")

def save_logs(logs):
    logs.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(logs_path) 

In [0]:

try:
    logger.info("===========Start Process===========")
    logger.info("Start reading silver table")
    df_silver = read_silver()
    logger.info("End reading silver table")

    logger.info("Start apply standard data quality for columns")
    df_fato_vendas = apply_refactory_name_column(df_silver)
    logger.info("End apply standard data quality for columns")

    logger.info("Start Ingestion fato_vendas")
    merge_fato_vendas(df_fato_vendas, columns_comments)
    logger.info("End Ingestion fato_vendas")
    
    logger.info("Start save logs")

    logs["status"] = "SUCCESS"
    logs["count_Rows"] = df_silver.count()
    logs["dat_Hr_Process_End"] = date_now()

    df_logs = spark.createDataFrame([logs])
    
    save_logs(df_logs)
    logger.info("End saving logs")
    logger.info("===========End Process===========")
except Exception as e:
    logs["status"] = "FAILED"
    logger.error(f'{e}')